# 🌳 Decision Trees
**ISLP Chapter 8 · Pattern Recognition for the Rest of Us**

> Decision trees split data by asking a sequence of yes/no questions. They're one of the most interpretable ML models — you can literally draw them out and explain them to anyone.

### What you'll learn
- How trees split using Gini impurity and RSS
- Growing and pruning trees to avoid overfitting
- Classification trees vs regression trees
- Tree visualization and interpretation
- Why trees are the building block for Random Forests and Boosting

### Dataset: Carseats (sales prediction) + Heart disease (classification)

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, mean_squared_error

In [ ]:
# ── Load datasets from the course repo ──────────────────────────────────────
import subprocess, os

# Clone the course repo if not already present (works in Colab)
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    # Fallback: already in repo root (e.g. running locally)
    DATA_DIR = '../data'

print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Available datasets: {os.listdir(DATA_DIR)}")

### Load Dataset: Carseats

In [ ]:
import pandas as pd
carseats = pd.read_csv(f'{DATA_DIR}/Carseats.csv')
carseats = pd.get_dummies(carseats, drop_first=True, dtype=float)
X_reg = carseats.drop('Sales', axis=1)
y_reg = carseats['Sales']
print(f"Carseats: {X_reg.shape}  |  Predicting: Sales")

### Load Dataset: Heart

In [ ]:
import pandas as pd
heart = pd.read_csv(f'{DATA_DIR}/Heart.csv').dropna()
heart = pd.get_dummies(heart, drop_first=True, dtype=float)
target_col = [c for c in heart.columns if 'AHD' in c or c == heart.columns[-1]][0]
X = heart.drop(target_col, axis=1)
y = heart[target_col].astype(int)
print(f"Heart: {X.shape}  |  Disease rate: {y.mean():.1%}")

## 🌳 Part 1 — How Trees Split

**For regression trees:** At each node, find the feature j and threshold s that minimize RSS:
```
RSS = Σ(yᵢ - ȳ_left)² + Σ(yᵢ - ȳ_right)²
```

**For classification trees:** Minimize **Gini impurity** at each node:
```
Gini = 1 - Σₖ p̂ₖ²
```
where p̂ₖ is the proportion of class k observations in the node.
A node with all one class has Gini=0 (pure). A 50/50 split has Gini=0.5 (most impure).

In [ ]:
# Visualize Gini impurity
p = np.linspace(0.001, 0.999, 200)
gini = 1 - p**2 - (1-p)**2
entropy = -p*np.log2(p) - (1-p)*np.log2(1-p)
misclass = 1 - np.maximum(p, 1-p)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(p, gini,     color='#1e5fa8', lw=2, label='Gini impurity')
ax.plot(p, entropy/2, color='#e85d20', lw=2, ls='--', label='Entropy/2')
ax.plot(p, misclass, color='#888',    lw=1.5, ls=':', label='Misclassification rate')
ax.set_xlabel('Proportion of class 1 in node')
ax.set_ylabel('Impurity measure')
ax.set_title('Node Impurity Measures — all maximized at 50/50 split')
ax.legend()
ax.axvline(0.5, color='black', lw=0.8, ls='--', alpha=0.4)
plt.tight_layout()
plt.show()
print("📌 Trees try to create pure nodes (low Gini). A pure node = all observations are one class.")

In [ ]:
# Grow and visualize a classification tree
X_tr, X_te, y_tr, y_te = train_test_split(X_clf, y_clf, test_size=0.3, random_state=42)

tree_clf = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)
tree_clf.fit(X_tr, y_tr)

print(f"Training accuracy: {tree_clf.score(X_tr, y_tr):.3f}")
print(f"Test accuracy:     {tree_clf.score(X_te, y_te):.3f}")
print(f"Number of leaves:  {tree_clf.get_n_leaves()}")

fig, ax = plt.subplots(figsize=(20, 7))
plot_tree(tree_clf, feature_names=X_clf.columns, class_names=['No Disease','Disease'],
          filled=True, rounded=True, fontsize=8, ax=ax,
          impurity=True, proportion=False)
ax.set_title('Classification Tree (max_depth=4) — Heart Disease Prediction', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# The overfitting problem — training vs test accuracy by tree depth
depths = range(1, 20)
train_acc, test_acc = [], []

for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=42)
    t.fit(X_tr, y_tr)
    train_acc.append(t.score(X_tr, y_tr))
    test_acc.append(t.score(X_te, y_te))

best_depth = depths[test_acc.index(max(test_acc))]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(depths), train_acc, 'o-', color='#1e5fa8', lw=2, label='Training accuracy')
ax.plot(list(depths), test_acc,  's-', color='#e85d20', lw=2, label='Test accuracy')
ax.axvline(best_depth, color='#1a7a45', lw=1.5, ls='--', label=f'Best depth = {best_depth}')
ax.set_xlabel('Max Tree Depth')
ax.set_ylabel('Accuracy')
ax.set_title('Overfitting: Deep Trees Memorize Training Data')
ax.legend()
plt.tight_layout()
plt.show()
print(f"\n📌 Training accuracy → 100% as depth increases (memorizes every point)")
print(f"   Test accuracy peaks at depth {best_depth} then falls — overfitting begins")

In [ ]:
# Regression tree on Carseats
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)
tree_reg = DecisionTreeRegressor(max_depth=4, random_state=42)
tree_reg.fit(X_tr_r, y_tr_r)

train_mse = mean_squared_error(y_tr_r, tree_reg.predict(X_tr_r))
test_mse  = mean_squared_error(y_te_r, tree_reg.predict(X_te_r))

print(f"Regression Tree (max_depth=4) — Carseats Sales")
print(f"Training RMSE: {np.sqrt(train_mse):.3f}")
print(f"Test RMSE:     {np.sqrt(test_mse):.3f}")

# Feature importance
imp = pd.Series(tree_reg.feature_importances_, index=X_reg.columns).sort_values(ascending=False).head(8)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(imp.index, imp.values, color='#1e5fa8', edgecolor='white')
ax.set_ylabel('Feature Importance')
ax.set_title('Regression Tree — Feature Importance for Carseats Sales')
ax.set_xticklabels(imp.index, rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
answers = {
    "q1": "",  # What is Gini impurity and what value indicates a pure node?
    "q2": "",  # Why do very deep trees overfit?
    "q3": "",  # What is the difference between pruning and setting max_depth?
    "q4": "",  # Name one advantage and one disadvantage of decision trees vs linear models
    "q5": "",  # Why are decision trees the building block for Random Forests and Boosting?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("❌ Still empty: "+str(missing) if missing else "✅ Done! File → Save a copy in GitHub")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Decision Trees"
# ══════════════════════════════════════════════════════════════════════
# 🤖 AI-GRADED SUBMISSION
# ══════════════════════════════════════════════════════════════════════
#
# REQUIRED — fill in your GitHub username so your instructor can
# match your submission to your name:
#
GITHUB_USERNAME = ""   # ← e.g. "jsmith42"  — your github.com username
#
# ── ONE-TIME SETUP (do this once, works for all 30 notebooks) ─────────
#
# You need a FREE Gemini API key from Google AI Studio.
#
# ⚠️  IMPORTANT: Use your PERSONAL Gmail — NOT your university email.
#     Many universities block Google AI Studio on managed accounts.
#     A personal @gmail.com works everywhere and is always free.
#
# Steps:
#   1. Open a private/incognito browser window
#   2. Go to: aistudio.google.com
#   3. Sign in with your PERSONAL Gmail (@gmail.com)
#   4. Click "Get API key" → "Create API key" → copy it
#   5. Back in Colab: click the 🔑 icon in the LEFT SIDEBAR
#      → "+ Add new secret"
#      → Name:  GEMINI_API_KEY
#      → Value: paste your key here
#      → Toggle the switch ON
#   6. Re-run this cell
#
# Your key is saved to your Colab account — works across all notebooks.
# It is FREE — no credit card, no billing required.
#
# ── NO KEY? ────────────────────────────────────────────────────────────
# You still get completion-based feedback without a key.
# You just won't get AI analysis of your specific answers.
#
# ══════════════════════════════════════════════════════════════════════
# DO NOT EDIT BELOW THIS LINE
# ══════════════════════════════════════════════════════════════════════
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Add more detail to show your understanding."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with reasonable detail. "
                             "Add a free Gemini key for AI analysis of your specific answers."),
                "tip": "Get a free key at aistudio.google.com — use your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             "Complete the remaining questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Set GITHUB_USERNAME above to track submissions")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
print("\n" + "\u2500"*58)
print("  Checking submission...")
print("\u2500"*58)

if "answers" not in globals():
    print("  \u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    if username:
        print(f"  Student  : @{username}")
    else:
        print(f"  Student  : \u26a0\ufe0f  GITHUB_USERNAME not set")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)")
        print()
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell")
        print()
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be Save: File \u2192 Save a copy in GitHub \u2192 your fork")
    print()
